# Lab 5: Monte Carlo Integration

---

## Overview

This lab explores **Monte Carlo integration**, a powerful numerical method that uses random sampling to approximate definite integrals. This problem is a synthesis of two GPU patterns you've learned:

1. **Embarrassingly parallel**: Each sample point can be evaluated independently
2. **Reduction**: All sample contributions must be summed to produce the final result

Monte Carlo methods are widely used in physics simulations, financial modeling, computer graphics (ray tracing), and machine learning.

## Learning Objectives

By the end of this lab, you will be able to:

1. Understand Monte Carlo integration and its convergence properties ($1/\sqrt{n}$ error)
2. Recognize the "embarrassingly parallel + reduction" pattern
3. Analyze the performance bottleneck of atomic-based reduction
4. Compare atomic reduction vs. shared memory tree reduction
5. Understand numerical precision considerations in parallel accumulation

## 1. Monte Carlo Integration Theory

### The Algorithm

1. Generate $n$ random points $x_i$ uniformly distributed in $[a, b]$
2. Evaluate $f(x_i)$ at each point (given as input $y_i$)
3. Compute the average: $\bar{y} = \frac{1}{n} \sum_{i=1}^{n} y_i$
4. Multiply by interval width: $\text{result} = (b - a) \times \bar{y}$

### Why GPU?

| Property | Benefit for GPU |
|:---------|:----------------|
| Independent samples | No data dependencies between threads |
| Large sample count | More parallelism = better GPU utilization |
| Reduction step | Standard parallel reduction pattern |

### Convergence

The error decreases with more samples:

$$
\text{Error} \propto \frac{1}{\sqrt{n}}
$$

To halve the error, we need 4x more samples.

## 2. Kernel Implementation Analysis

Let us examine the kernel in `main.hip`:

```cpp
__global__ void montecarlo(const double* y_samples, double* result, 
                           double a, double b, int n_samples) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    if (idx >= n_samples) return;

    double tmp = (b - a) * y_samples[idx] / n_samples;

    atomicAdd(result, tmp); 
}
```

### Algorithm Breakdown

| Step | Operation | Parallelism |
|:-----|:----------|:------------|
| 1 | Load $y_i$ | Coalesced memory access |
| 2 | Compute $(b-a) \times y_i / n$ | Fully parallel |
| 3 | Accumulate to result | Atomic (serialized) |

### Exercise 1: Understand the Algorithm

**Question 1**: Why is the division by `n_samples` done in each thread instead of once at the end?

Your answer: It scales each sample contribution before the atomic accumulation, so the value added by every thread is already `(b - a) * y_i / n_samples`. Mathematically this is equivalent to summing all `y_i` first and multiplying once at the end. It also lets the kernel directly accumulate the final integral value in `result`, although doing the division once after reduction would usually reduce repeated arithmetic.

**Question 2**: What is the time complexity of the atomic reduction approach?

Your answer: The kernel launches `O(n_samples)` threads and each thread performs one atomic add. The total work is `O(n_samples)`, but the atomic updates contend for one global memory address, so the accumulation bottleneck is effectively serialized and scales poorly as `n_samples` grows.

**Question 3**: How could you modify this kernel to use shared memory reduction instead of atomics?

Your answer: Let each block first load or accumulate its assigned samples into a per-thread `local_sum`, store those partial sums in `__shared__` memory, then perform a tree reduction inside the block using `__syncthreads()`. After each block has one block-level sum, only thread 0 of the block performs one global `atomicAdd` to `result`. This reduces global atomics from one per sample to one per block.


## 3. Setup: Generate Test Data

In [ ]:
%%bash
# Generate test cases
python3 geninput.py
echo "Test cases generated:"
ls -la testcases/

## 4. Compile the Program

In [ ]:
%%bash
# Compile the Monte Carlo integration program
hipcc -O2 fs_main.hip -o exe_montecarlo
echo "Compilation complete."

## 5. Run Small Test Cases

In [ ]:
%%bash
echo "=== Test 1: Small sample (N=8) ==="
cat testcases/1.in
echo "Result:"
./exe_montecarlo testcases/1.in

In [ ]:
%%bash
echo "=== Test 2: Single sample (N=1) ==="
cat testcases/2.in
echo "Result:"
./exe_montecarlo testcases/2.in

### Exercise 2: Manual Calculation

For Test 1, verify the result manually:

Given: $a = 0$, $b = 2$, $n = 8$ samples

Formula: $\text{result} = \frac{(b-a)}{n} \sum_{i=1}^{n} y_i = \frac{2}{8} \times \sum y_i$

Your calculation:
- Sum of y values: `4.1805 + 1.7864 + 4.7554 + 3.9983 + 2.2786 + 1.1873 + 4.1222 + 1.4420 = 23.7507`
- Expected result: `(2 / 8) * 23.7507 = 5.937675`
- Does it match the program output? It should match the generated Test 1 output, up to the displayed floating-point formatting.


## 6. Performance Scaling

In [ ]:
%%bash
echo "=== Test 4: N=100,000 ==="
time ./exe_montecarlo testcases/4.in

In [ ]:
%%bash
echo "=== Test 7: N=10,000,000 ==="
time ./exe_montecarlo testcases/7.in

In [ ]:
%%bash
echo "=== Test 8: N=100,000,000 ==="
time ./exe_montecarlo testcases/8.in

### Exercise 3: Scalability Analysis

Record the execution times:

| Test | Sample Count | Execution Time |
|:-----|:-------------|:---------------|
| 4 | 100,000 | `real 0m0.074s` |
| 7 | 10,000,000 | `real 0m0.878s` |
| 8 | 100,000,000 | `real 0m8.425s` |

**Question 1**: When sample count increases 100x (from test 4 to 7), how much does execution time increase?

Your answer: The execution time increases from `0.074 s` to `0.878 s`, so the increase is `0.878 / 0.074 = 11.86x`, about `11.9x`. From Test 7 to Test 8, the sample count increases by `10x`, while the runtime increases from `0.878 s` to `8.425 s`, which is `8.425 / 0.878 = 9.60x`. This second ratio is much closer to linear because the larger cases are dominated by input processing, device memory traffic, and kernel execution rather than fixed startup overhead.

**Question 2**: Is the scaling linear? Why or why not?

Your answer: The scaling is not perfectly linear. For small input sizes, fixed overheads such as file I/O, memory allocation, host-to-device copy, kernel launch, and synchronization occupy a large fraction of the total time, so increasing the sample count does not produce a proportional runtime increase. For larger inputs, the runtime becomes closer to linear because most time is spent moving and processing the sample array. However, the current kernel still uses one global `atomicAdd` per sample, so many threads compete for the same global memory location. This atomic contention prevents ideal parallel scaling and is the main reason a shared-memory reduction version would be preferred.


## 7. Embarrassingly Parallel vs. Reduction

The Monte Carlo kernel has two distinct phases:

### Phase 1: Embarrassingly Parallel

```cpp
double tmp = (b - a) * y_samples[idx] / n_samples;
```

- Each thread computes independently
- No synchronization needed
- Scales perfectly with GPU cores

### Phase 2: Reduction (Bottleneck)

```cpp
atomicAdd(result, tmp);
```

- All threads compete for single location
- Serialized execution
- Does not scale with more threads

### Exercise 4: Bottleneck Analysis

**Question 1**: If the computation takes 1 cycle and atomic takes 100 cycles, what is the theoretical efficiency for N=1M threads?

Hint: Total work = N x (compute + atomic)

Your calculation: Useful compute per thread is `1` cycle and total per-thread cost is approximately `1 + 100 = 101` cycles, so the theoretical efficiency is `1 / 101 = 0.0099`, about `0.99%`. For `N = 1,000,000`, that is `1,000,000` useful compute cycles out of about `101,000,000` total cycles. This simple model shows that almost all time is spent on atomic accumulation rather than on the Monte Carlo contribution calculation itself.

**Question 2**: How would shared memory reduction improve this?

Your answer: Shared memory reduction performs most additions inside each block using fast shared memory and synchronization. Instead of every thread updating the same global address, each block reduces its own partial sum locally and then performs a single global atomic update. This changes the number of global atomics from approximately `N` to approximately `ceil(N / blockDim.x)`, so the expensive serialized part is greatly reduced. The result is better scalability, lower global memory contention, and usually better numerical behavior because tree reduction adds values in a more balanced order.

**Question 3**: With 256 threads per block and N=1M, how many global atomics would shared memory reduction require?

Your calculation: The number of blocks is `ceil(1,000,000 / 256) = 3,907`, so the optimized reduction needs `3,907` global atomics instead of `1,000,000`. This is about `1,000,000 / 3,907 = 255.95x` fewer global atomic operations, which is nearly a `256x` reduction in atomic pressure.


## 8. Numerical Precision

### Floating Point Accumulation

Adding many small numbers to a large sum can cause precision loss:

```
sum = 1000000.0
sum += 0.0001  // May be lost due to precision
```

### Why Tree Reduction Helps

| Method | Accumulation Pattern | Precision |
|:-------|:--------------------|:----------|
| Sequential | Large + small + small + ... | Poor |
| Tree | (a+b) + (c+d) + ... | Better |

### Exercise 5: Precision Considerations

**Question 1**: The kernel uses `double` (64-bit). How many significant digits does double precision provide?

Your answer: IEEE 754 double precision provides about 15 to 17 decimal significant digits, with a 53-bit significand including the implicit leading bit.

**Question 2**: If we switch to `float` (32-bit), what problems might occur with N=100M samples?

Your answer: Float precision provides only about 6 to 7 decimal significant digits. With `N = 100,000,000`, many small contributions may be rounded away during accumulation, especially when the running sum is large. The final integral may have noticeably larger rounding error, and the result may depend more strongly on the order of parallel accumulation.


## 9. Optimized Implementation Design

### Two-Phase Approach

```cpp
__global__ void montecarlo_optimized(const double* y_samples, double* result,
                                      double a, double b, int n_samples) {
    __shared__ double sdata[256];
    
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Phase 1: Each thread accumulates multiple elements
    double local_sum = 0.0;
    for (int i = idx; i < n_samples; i += blockDim.x * gridDim.x) {
        local_sum += y_samples[i];
    }
    sdata[tid] = local_sum;
    __syncthreads();
    
    // Phase 2: Tree reduction in shared memory
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            sdata[tid] += sdata[tid + s];
        }
        __syncthreads();
    }
    
    // Only one atomic per block
    if (tid == 0) {
        atomicAdd(result, (b - a) * sdata[0] / n_samples);
    }
}
```

### Exercise 6: Optimization Analysis

**Question 1**: In the optimized version, how many global atomics occur for N=1M with 256 blocks?

Your answer: `256` global atomics, because each block produces one partial sum and only thread 0 in each block performs one `atomicAdd`. Compared with the original one-atomic-per-sample kernel, this reduces the atomic count from `1,000,000` to `256`, or about `3906x` fewer global atomic operations.

**Question 2**: What is the purpose of the grid-stride loop `for (int i = idx; i < n_samples; i += blockDim.x * gridDim.x)`?

Your answer: The grid-stride loop lets a fixed number of launched threads cover any number of input samples. Each thread processes multiple elements separated by the total number of threads in the grid. This avoids requiring one GPU thread for every sample when `n_samples` is very large, improves occupancy control, and keeps memory access coalesced within each loop iteration because neighboring threads still read neighboring elements.

**Question 3**: Why is `(b - a) / n_samples` multiplication done only by thread 0?

Your answer: The block first reduces raw `y_i` values into one block sum. Applying `(b - a) / n_samples` once to the block sum avoids repeating the same scale operation for every sample and reduces arithmetic overhead. It also keeps the reduction focused on summing sample values, then converts the final block-level sum into an integral contribution only when writing the block result to global memory.


## 10. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **Monte Carlo Integration** | Approximate integrals using random sampling |
| **Embarrassingly Parallel** | Independent computations with no data dependencies |
| **Reduction** | Combining many values into one |
| **Atomic Contention** | Performance bottleneck when many threads update same location |

### GPU Suitability

| Aspect | GPU Advantage |
|:-------|:--------------|
| Sample evaluation | Thousands of parallel evaluations |
| Large N required for accuracy | More work = better GPU utilization |
| Memory bandwidth | Coalesced reads of sample array |

### Optimization Takeaways

1. Use shared memory to reduce atomic contention
2. Each thread should process multiple elements
3. Tree reduction improves both performance and precision
4. Number of global atomics = number of blocks (not threads)

## 11. Challenge Exercises

### Challenge 1: Implement Shared Memory Reduction

Modify `main.hip` to use the two-phase approach shown in Section 9. Compare performance.

### Challenge 2: Compute Pi Using Monte Carlo

Classic application: estimate $\pi$ by sampling points in a square and counting how many fall inside a quarter circle.

$$
\pi \approx 4 \times \frac{\text{points inside circle}}{\text{total points}}
$$

### Challenge 3: Error Analysis

For a known integral (e.g., $\int_0^1 x^2 dx = 1/3$):
1. Run with N = 1000, 10000, 100000, 1000000
2. Calculate absolute error for each
3. Verify the $1/\sqrt{n}$ convergence rate

### Challenge 4: Compare float vs double

Modify the kernel to use `float` instead of `double`. Test with large N and compare:
- Accuracy of result
- Execution time

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `main.hip` | Student implementation (stdin input) |
| `fs_main.hip` | Reference implementation (file input) |
| `geninput.py` | Test case generator |
| `Makefile` | Build configuration |